# Minería de texto: similitud semántica mediante embeddings

## Contexto práctico

Una mesa de ayuda recibe muchos tickets. Antes de resolver un ticket nuevo, un agente debe buscar manualmente si ya existe un caso parecido y revisar cómo se solucionó.

En este ejercicio construiremos un buscador semántico. El usuario escribirá una consulta nueva y el sistema devolverá los tickets históricos más parecidos, junto con su solución registrada.

La ventaja es que podemos encontrar textos con significado similar aunque no utilicen exactamente las mismas palabras.

## Objetivos de aprendizaje

Al ejecutar el notebook podrá:

- explicar qué es un embedding;
- diferenciar similitud semántica de TF-IDF y NER;
- convertir textos en vectores numéricos;
- calcular similitud coseno;
- recuperar los casos históricos más parecidos;
- establecer un umbral para aceptar o revisar una coincidencia;
- visualizar los documentos en un espacio de dos dimensiones;
- interpretar los resultados desde una perspectiva operativa.

El dataset está incluido como una lista de registros dentro del notebook. No tendrá que subir un archivo externo.

## ¿En qué se diferencia de los ejercicios anteriores?

- TF-IDF y regresión logística responden: ¿a qué categoría pertenece el texto?
- NER responde: ¿qué entidad aparece dentro del texto y dónde está?
- Embeddings y similitud semántica responden: ¿qué textos expresan una idea parecida?

Aquí no entrenaremos un clasificador con categorías. La tarea principal será recuperar información relevante desde un histórico de casos.

## 1. Instalar las bibliotecas

sentence-transformers proporciona modelos que convierten oraciones en embeddings. El modelo multilingüe utilizado funciona con español. La salida de instalación se captura para evitar mensajes técnicos innecesarios en el notebook.

In [ ]:
%%capture
!pip -q install sentence-transformers pandas matplotlib seaborn scikit-learn

## 2. Importar bibliotecas y cargar el modelo

SentenceTransformer carga un modelo previamente entrenado. No empezamos desde cero: el modelo ya aprendió relaciones semánticas a partir de grandes colecciones de texto.

normalize_embeddings=True deja los vectores normalizados. En ese caso, el producto punto entre dos vectores equivale a la similitud coseno.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

MODELO_EMBEDDING = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo = SentenceTransformer(MODELO_EMBEDDING)
print(f'Modelo cargado: {MODELO_EMBEDDING}')

## 3. Concepto: ¿qué es un embedding?

Un embedding es una representación numérica de un texto. En lugar de guardar una oración como una secuencia de palabras, el modelo la transforma en un vector de muchos números.

La idea central es que textos con significado parecido quedan cerca en el espacio vectorial. Por ejemplo, “no puedo entrar a mi cuenta” y “mi acceso está bloqueado” deberían tener vectores cercanos aunque no compartan todas las palabras.

El embedding representa el significado aprendido por el modelo; no es una traducción ni una lista de palabras importantes.

## 4. Dataset incluido en el notebook

Cada registro contiene un ticket histórico, su área y la solución aplicada. La categoría no será utilizada para calcular la similitud; se conservará para interpretar si los resultados recuperados pertenecen al mismo tipo de problema.

In [ ]:
datos = [
    {'id': 'T001', 'ticket': 'No puedo entrar a mi cuenta porque olvidé la contraseña', 'area': 'acceso', 'solucion': 'Restablecer la contraseña desde el enlace de recuperación y confirmar el correo.'},
    {'id': 'T002', 'ticket': 'Mi usuario quedó bloqueado después de varios intentos', 'area': 'acceso', 'solucion': 'Desbloquear el usuario y solicitar una nueva contraseña temporal.'},
    {'id': 'T003', 'ticket': 'El código de verificación no llega al correo', 'area': 'acceso', 'solucion': 'Validar el correo registrado y reenviar el código de autenticación.'},
    {'id': 'T004', 'ticket': 'La aplicación se cierra cuando intento iniciar sesión', 'area': 'acceso', 'solucion': 'Actualizar la aplicación y limpiar la caché del dispositivo.'},
    {'id': 'T005', 'ticket': 'El paquete todavía no llega a mi domicilio', 'area': 'entrega', 'solucion': 'Consultar la guía y escalar el envío con la empresa de paquetería.'},
    {'id': 'T006', 'ticket': 'Quiero rastrear mi pedido porque aparece retrasado', 'area': 'entrega', 'solucion': 'Compartir el número de guía y la fecha estimada de entrega.'},
    {'id': 'T007', 'ticket': 'El pedido llegó incompleto', 'area': 'entrega', 'solucion': 'Solicitar evidencia y generar un reporte por faltante de productos.'},
    {'id': 'T008', 'ticket': 'Recibí el paquete dañado durante la entrega', 'area': 'entrega', 'solucion': 'Registrar evidencia fotográfica e iniciar el proceso de reposición.'},
    {'id': 'T009', 'ticket': 'No reconozco un cargo en mi factura', 'area': 'facturacion', 'solucion': 'Revisar el detalle de cargos y abrir una aclaración de pago.'},
    {'id': 'T010', 'ticket': 'Me cobraron dos veces el mismo servicio', 'area': 'facturacion', 'solucion': 'Validar la transacción duplicada y gestionar el reembolso.'},
    {'id': 'T011', 'ticket': 'Necesito una factura con mis datos fiscales', 'area': 'facturacion', 'solucion': 'Actualizar los datos fiscales y emitir una factura corregida.'},
    {'id': 'T012', 'ticket': 'El importe de la factura no coincide con mi contrato', 'area': 'facturacion', 'solucion': 'Comparar el contrato con el cobro y corregir la diferencia.'},
    {'id': 'T013', 'ticket': 'Quiero cancelar mi suscripción mensual', 'area': 'cancelacion', 'solucion': 'Confirmar identidad y procesar la baja antes del siguiente cobro.'},
    {'id': 'T014', 'ticket': 'Ya no deseo continuar con el plan contratado', 'area': 'cancelacion', 'solucion': 'Registrar la solicitud de cancelación y enviar confirmación.'},
    {'id': 'T015', 'ticket': 'Solicito eliminar definitivamente mi cuenta', 'area': 'cancelacion', 'solucion': 'Validar identidad y ejecutar el proceso de eliminación de cuenta.'},
    {'id': 'T016', 'ticket': 'Necesito terminar el contrato del servicio', 'area': 'cancelacion', 'solucion': 'Revisar condiciones del contrato y confirmar la fecha efectiva de baja.'},
    {'id': 'T017', 'ticket': 'La página muestra un error cuando guardo los cambios', 'area': 'soporte_tecnico', 'solucion': 'Revisar los registros del sistema y reproducir el error.'},
    {'id': 'T018', 'ticket': 'El portal está muy lento desde esta mañana', 'area': 'soporte_tecnico', 'solucion': 'Revisar disponibilidad del servicio y métricas de rendimiento.'},
    {'id': 'T019', 'ticket': 'El botón de pago no responde', 'area': 'soporte_tecnico', 'solucion': 'Verificar el navegador, consola de errores y disponibilidad del proveedor de pagos.'},
    {'id': 'T020', 'ticket': 'La aplicación se queda cargando y no muestra información', 'area': 'soporte_tecnico', 'solucion': 'Comprobar conexión, limpiar caché y revisar la disponibilidad del servicio.'}
]

df = pd.DataFrame(datos)
df.to_csv('tickets_historicos_embeddings.csv', index=False, encoding='utf-8')
print(f'Tickets históricos: {len(df)}')
display(df.head())

### Interpretación del dataset

El histórico funciona como una base de conocimiento sencilla. Cada ticket aporta una experiencia previa y una solución que podría ayudar a resolver una consulta nueva.

El sistema no aprende una regla como “si aparece paquete, elegir entrega”. Busca cercanía semántica entre el nuevo texto y todos los textos históricos.

## 5. Generar embeddings para los tickets

La siguiente celda transforma cada ticket en un vector. El resultado tendrá una fila por ticket y muchas columnas numéricas. No es necesario interpretar cada número individualmente; lo importante es comparar vectores completos.

In [ ]:
embeddings_tickets = modelo.encode(
    df['ticket'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False
)

print(f'Matriz de embeddings: {embeddings_tickets.shape}')
print('Cada fila representa un ticket y cada columna una dimensión semántica.')

## 6. Buscar tickets similares

La función recibe una consulta, genera su embedding, calcula la similitud coseno contra todos los tickets y ordena los resultados de mayor a menor.

La similitud coseno suele interpretarse así: valores cercanos a 1 indican alta cercanía de dirección semántica; valores cercanos a 0 indican poca relación. El valor exacto depende del modelo y debe calibrarse con datos reales.

In [ ]:
def buscar_similares(consulta, top_n=3):
    embedding_consulta = modelo.encode([consulta], normalize_embeddings=True)
    similitudes = cosine_similarity(embedding_consulta, embeddings_tickets)[0]
    indices = similitudes.argsort()[::-1][:top_n]
    resultado = df.iloc[indices][['id', 'ticket', 'area', 'solucion']].copy()
    resultado.insert(1, 'similitud', similitudes[indices].round(3))
    return resultado.reset_index(drop=True)

consulta = 'Sigo esperando mi envío y necesito saber dónde está'
resultado = buscar_similares(consulta, top_n=3)
print(f'Consulta: {consulta}')
display(resultado)

### Interpretación del primer resultado

El primer registro es la recomendación principal. Debemos revisar dos cosas: el valor de similitud y el contenido de la solución.

Una similitud alta indica que el ticket se parece semánticamente, pero no garantiza que la solución sea correcta. El agente debe confirmar que el contexto, la política y los datos del cliente coincidan antes de reutilizarla.

## 7. Probar varias consultas prácticas

Evaluaremos consultas con redacciones diferentes a las del histórico. Esto es importante porque el valor de los embeddings aparece cuando no dependemos de repetir exactamente las mismas palabras.

In [ ]:
consultas = [
    'No puedo acceder porque mi clave ya no funciona',
    'El cobro de mi cuenta aparece duplicado',
    'Mi paquete sigue perdido y quiero localizarlo',
    'Necesito dar de baja el plan',
    'El sistema no responde cuando intento guardar'
]

for consulta in consultas:
    top = buscar_similares(consulta, top_n=1).iloc[0]
    print(f'Consulta: {consulta}')
    print(f"  Mejor caso: {top['id']} | Área: {top['area']} | Similitud: {top['similitud']}")
    print(f"  Solución sugerida: {top['solucion']}\n")

### Cómo interpretar estas búsquedas

Compare la consulta nueva con el ticket recuperado, no solamente con su área. Una buena coincidencia conserva la intención principal y el tipo de problema.

Si la similitud es baja o la solución no corresponde, el sistema debe informar que no encontró un antecedente confiable. Es preferible solicitar revisión humana que recomendar una solución incorrecta.

## 8. Umbral de confianza operacional

La similitud no debe usarse como un sí o no universal. Definiremos una regla sencilla: si el mejor resultado supera 0.55, se muestra como antecedente; de lo contrario, se marca para revisión.

El valor 0.55 es únicamente didáctico. En un proyecto real se elegiría usando ejemplos históricos evaluados por agentes.

In [ ]:
UMBRAL = 0.55

def recomendar_solucion(consulta, umbral=UMBRAL):
    top = buscar_similares(consulta, top_n=1).iloc[0]
    if top['similitud'] >= umbral:
        estado = 'Antecedente suficientemente parecido; revisar antes de aplicar.'
    else:
        estado = 'Coincidencia débil; enviar a revisión humana.'
    return pd.DataFrame([{
        'consulta': consulta,
        'ticket_sugerido': top['id'],
        'similitud': top['similitud'],
        'area': top['area'],
        'estado': estado,
        'solucion': top['solucion']
    }])

display(recomendar_solucion('Mi envío no ha llegado y quiero rastrearlo'))
display(recomendar_solucion('Quiero cambiar el color de mi perfil'))

### Interpretación del umbral

La primera consulta debería recuperar un caso de entrega y puede servir como apoyo al agente. La segunda no tiene un antecedente claro en el dataset, por lo que debe revisarse manualmente.

El umbral controla el equilibrio entre cobertura y seguridad: un umbral bajo devuelve más recomendaciones, pero aumenta el riesgo de coincidencias irrelevantes; uno alto es más conservador y puede dejar más casos sin sugerencia.

## 9. Visualizar los embeddings

Los embeddings tienen muchas dimensiones y no pueden graficarse directamente. PCA los proyecta en dos dimensiones para obtener una vista aproximada de las relaciones.

La gráfica no representa todo el significado del modelo. Sirve como herramienta exploratoria: puntos cercanos suelen ser casos parecidos, pero una separación visual no reemplaza la evaluación numérica.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coordenadas = pca.fit_transform(embeddings_tickets)
grafica_df = df.copy()
grafica_df['componente_1'] = coordenadas[:, 0]
grafica_df['componente_2'] = coordenadas[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=grafica_df, x='componente_1', y='componente_2', hue='area', s=110)
for _, fila in grafica_df.iterrows():
    plt.text(fila['componente_1'] + 0.01, fila['componente_2'] + 0.01, fila['id'], fontsize=8)
plt.title('Proyección PCA de los tickets históricos')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.show()
print(f'Varianza explicada por las dos dimensiones: {pca.explained_variance_ratio_.sum():.1%}')

### Interpretación de la visualización

Si los tickets de una misma área aparecen relativamente cercanos, significa que comparten señales semánticas. Si hay solapamiento, es normal: problemas como una aplicación que no carga pueden relacionarse con acceso y soporte técnico.

PCA puede perder información al reducir dimensiones. Por eso no debemos afirmar que dos documentos son iguales solo porque se ven cercanos en la gráfica.

## 10. Conclusiones generales

1. Los embeddings representan el significado global de un texto en un vector numérico.
2. La similitud coseno permite recuperar casos parecidos aunque usen palabras diferentes.
3. Esta técnica es adecuada para búsqueda semántica, tickets duplicados, preguntas frecuentes y recuperación de soluciones.
4. Una similitud alta es una recomendación, no una garantía: la solución debe revisarse antes de aplicarse.
5. El umbral debe calibrarse con casos reales y con el costo de recomendar una solución incorrecta.
6. La visualización ayuda a explorar patrones, pero la evaluación debe basarse en ejemplos etiquetados y métricas operativas.
7. El sistema mejora cuando el histórico contiene soluciones correctas, variadas y actualizadas.

### Conclusión ejecutiva

La similitud semántica mediante embeddings permite transformar una base de tickets históricos en un asistente de búsqueda para agentes. Su utilidad práctica está en reducir el tiempo de búsqueda y reutilizar conocimiento existente, manteniendo revisión humana para los casos dudosos.